# USD/CAD — Your Starter Agent

This notebook implements the starter agent for the USD/CAD forecasting use case.

**Target:** `usdcad_logret_{h}b = log(DEXCAUS[t+h] / DEXCAUS[t])`.

Because FRED `DEXCAUS` is Canadian dollars per U.S. dollar, a positive target return means USD/CAD rises (USD strengthens relative to CAD); a negative return means CAD strengthens relative to USD.

The agent supports cutoff-aware web research, optional code diagnostics, and the leak-safe FRED covariate panel already implemented in this use case.

In [2]:
import warnings
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import sys
warnings.filterwarnings("ignore")

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return here


ROOT = _repo_root()
IMPLEMENTATIONS = ROOT / "implementations"
if str(IMPLEMENTATIONS) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATIONS))
load_dotenv(ROOT / ".env", override=False)  # LLMP rows call the Vector proxy — need PROXY_* set

AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"
RUN_AGENT = True

from usdcad_forecasting import DEFAULT_COVARIATE_SERIES_IDS
from usdcad_forecasting.starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)

print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)
print("Default covariates:", DEFAULT_COVARIATE_SERIES_IDS)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview
Default covariates: ['usdcad_log_ret_1b_l1b', 'fed_funds_level_l1b', 'ust2y_level_l1b', 'ust10y_level_l1b', 'ust2y10y_spread_l1b', 'cpi_mom_logdiff_l1b', 'unemployment_rate_l1b', 'oil_log_ret_1b_l1b']


---
## 1. Meet your agent

Search is enabled by default; code execution is optional. The forecasting skill is always loaded.

In [3]:
config = build_starter_agent_config(
    model=AGENT_MODEL,
    enable_search=True,
    enable_code_exec=False,
)

print("Agent:", config.name)
print("Search enabled:", config.context_retrieval.enabled)
print("Code-exec enabled:", config.code_execution.enabled)
print("Skills loaded:", [p.name for p in config.skills_dirs])
print("\nSystem instruction:\n")
print(config.instruction)

Agent: usdcad_starter_agent
Search enabled: True
Code-exec enabled: False
Skills loaded: ['forecasting', 'research-playbook']

System instruction:

## Role

You are a foreign-exchange analyst specializing in USD/CAD. You understand Bank of Canada and Federal Reserve policy paths, US-Canada rate differentials, Canadian and US macro data, WTI/oil, commodity sensitivity, risk sentiment, and CAD-specific drivers. Keep reasoning transparent and avoid false precision.

## How to respond

- For open-ended questions, answer directly and concisely; do not ask for a JSON payload.
- For a structured probabilistic forecast, use the forecasting skill and produce a calibrated distribution.
- DEXCAUS is Canadian dollars per US dollar: positive return means USD/CAD rises and USD strengthens relative to CAD.
- Separate observed evidence from interpretation and never use information after the forecast cutoff.


---
## 2. Talk to it — USD/CAD market analysis

Use this mode for exploratory FX questions. It does not require the structured forecast output schema.

In [4]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig

QUESTION = (
    "What are the main USD/CAD drivers right now? Discuss Bank of Canada and "
    "Federal Reserve policy, US-Canada rate differentials, oil, and Canadian/US "
    "macro data. Keep it concise."
)

if RUN_AGENT:
    chat_agent = build_adk_agent(config)
    runner = AdkTextRunner(
        chat_agent,
        config=AdkTextRunnerConfig(app_name="usdcad_starter_chat"),
    )
    reply = await runner.run_text_async(QUESTION)
    print(reply)
else:
    print("RUN_AGENT is False — set it to True to talk to the agent.")

As of September 23, 2026, the USD/CAD is primarily driven by a widening interest rate gap, intense trade uncertainty, and a resilient U.S. economy relative to Canada’s tariff-constrained outlook.

### Key Drivers

*   **Policy Path & Rate Differentials:** The Federal Reserve hiked rates to 3.75%–4.00% on September 16, reinforcing a restrictive bias. In contrast, the Bank of Canada (BoC) has held at 2.25%. The resulting 148-basis-point yield differential in favor of the USD serves as the primary fundamental anchor for USD strength.
*   **Trade Policy:** The breakdown of Canada-U.S. trade talks and the imposition of tariffs (steel, aluminum, auto) are significant drags on the CAD. These frictions are actively cooling Canadian growth projections for Q4, offsetting the benefit of generally high oil prices.
*   **Macroeconomic Divergence:** The U.S. continues to exhibit strong productivity and investment, supporting the dollar. Conversely, the Canadian economy, while posting solid Q2 growth

Root node usdcad_starter_agent was cancelled.


---
## 3. Score one resolved forecast

The cell below forecasts a single resolved origin. Start with 5 business days, then repeat for 1 and 21.

In [5]:
from datetime import datetime, timezone
from aieng.forecasting.evaluation.task import ForecastingTask
from usdcad_forecasting import (
    build_usdcad_multivariate_service,
    usdcad_logret_series_id,
)

HORIZON = 5
COVARIATES = DEFAULT_COVARIATE_SERIES_IDS

if RUN_AGENT:
    svc = build_usdcad_multivariate_service(
        covariate_series_ids=COVARIATES,
    )
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    tgt = usdcad_logret_series_id(HORIZON)
    full = svc.get_series(tgt, as_of=now)
    full["timestamp"] = pd.to_datetime(full["timestamp"])
    last_date = full["timestamp"].iloc[-1]

    # Pick the most recent origin with a fully resolved H-day target.
    AS_OF = last_date - pd.offsets.BDay(HORIZON + 1)

    task = ForecastingTask(
        task_id=f"usdcad_logret_{HORIZON}b",
        target_series_id=tgt,
        horizons=[HORIZON],
        frequency="B",
        description=f"USD/CAD cumulative log return, {HORIZON} business days ahead.",
    )

    ctx = svc.context(as_of=AS_OF)
    pred = build_starter_agent_predictor(
        config,
        covariate_series_ids=COVARIATES,
    ).predict(task, ctx)[0]

    rows = full[full["timestamp"] >= AS_OF + pd.offsets.BDay(HORIZON)]
    actual = float(rows["value"].iloc[0]) if not rows.empty else None

    fc = pred.payload
    lo = fc.quantiles[0.10]
    hi = fc.quantiles[0.90]

    print(
        f"Origin as_of={AS_OF.date()} | horizon={HORIZON}b | "
        f"latest target data={last_date.date()}\n"
    )
    print(
        f"  agent point  : {fc.point_forecast:+.6f} "
        f"({fc.point_forecast * 100:+.3f}%)"
    )
    print(f"  agent 80% CI : [{lo:+.6f}, {hi:+.6f}]")

    if actual is None:
        print("  actual       : N/A")
    else:
        in_band = "yes ✓" if lo <= actual <= hi else "no ✗"
        print(
            f"  actual       : {actual:+.6f} "
            f"({actual * 100:+.3f}%) | in 80% band? {in_band}"
        )

    if pred.metadata.get("rationale"):
        print("\nRationale:", pred.metadata["rationale"][:500])
else:
    print("RUN_AGENT is False — set it to True to score a live forecast.")

Origin as_of=2026-08-20 | horizon=5b | latest target data=2026-08-28

  agent point  : -0.005500 (-0.550%)
  agent 80% CI : [-0.018500, +0.007500]
  actual       : -0.005211 (-0.521%) | in 80% band? yes ✓

Rationale: The forecast reflects a balance between the divergence in central bank paths (USD support) and short-term mean reversion and commodity support (CAD support). Uncertainties remain elevated due to US-Canada trade frictions.


---
## 4. Try the three horizons

Use `HORIZON = 1`, `5`, or `21`. These map to `usdcad_logret_1b`, `usdcad_logret_5b`, and `usdcad_logret_21b`.

Recommended progression:
1. Target-only agent.
2. Agent + default FRED covariates.
3. Agent + cutoff-aware search.
4. Agent + search + code diagnostics.
5. Compare configurations across the multivariate backtest.

## 5. Covariates

The current default panel contains lagged USD/CAD return, US effective fed funds, US 2Y and 10Y yields, the 2Y–10Y spread, CPI growth, unemployment, and WTI return. These are constructed by the existing data module with its anti-leakage policy.

In [20]:
# Cell: Experiment setup

from pathlib import Path
import pandas as pd

from starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)

AGENT_MODEL = "gemini-3.1-flash-lite-preview"

HORIZON = 5

# Keep this False while developing/testing.
RUN_AGENT = False

# Cell: Build the three experimental agents

EXPERIMENTS = {
    "baseline": {
        "description": "Target history only",
        "enable_search": False,
        "enable_code_exec": False,
        "covariates": [],
    },
    "fred_covariates": {
        "description": "Target history + FRED covariates",
        "enable_search": False,
        "enable_code_exec": False,
        "covariates": DEFAULT_COVARIATE_SERIES_IDS,
    },
    "agentic_search": {
        "description": "Target history + FRED covariates + search",
        "enable_search": True,
        "enable_code_exec": False,
        "covariates": DEFAULT_COVARIATE_SERIES_IDS,
    },
}

agents = {}

for name, settings in EXPERIMENTS.items():

    config = build_starter_agent_config(
        model=AGENT_MODEL,
        enable_search=settings["enable_search"],
        enable_code_exec=settings["enable_code_exec"],
    )

    agents[name] = build_starter_agent_predictor(
        config,
        covariate_series_ids=settings["covariates"],
    )

    print(
        f"{name}: "
        f"search={settings['enable_search']}, "
        f"covariates={len(settings['covariates'])}"
    )

baseline: search=False, covariates=0
fred_covariates: search=False, covariates=8
agentic_search: search=True, covariates=8


In [22]:
# Cell: Create ForecastContext

from datetime import datetime, timezone

from usdcad_forecasting import (
    build_usdcad_multivariate_service,
)

HORIZON = 5

# The service needs the complete covariate panel because
# some of the agents will use it.
svc = build_usdcad_multivariate_service(
    covariate_series_ids=DEFAULT_COVARIATE_SERIES_IDS
)

now = datetime.now(tz=timezone.utc).replace(tzinfo=None)

target_series_id = f"usdcad_logret_{HORIZON}b"

full = svc.get_series(
    target_series_id,
    as_of=now,
)

full["timestamp"] = pd.to_datetime(full["timestamp"])

last_date = full["timestamp"].iloc[-1]

# Choose a historical origin whose HORIZON-day outcome has resolved.
AS_OF = last_date - pd.offsets.BDay(HORIZON + 1)

# THIS is the object that predictor.predict() expects.
ctx = svc.context(
    as_of=AS_OF
)

print("Context type:", type(ctx))
print("Forecast origin:", AS_OF.date())
print("Latest data:", last_date.date())

Context type: <class 'aieng.forecasting.data.context.ForecastContext'>
Forecast origin: 2026-08-20
Latest data: 2026-08-28


In [9]:
# Cell: Load USDCAD forecasting data

from usdcad_forecasting.data import (
    build_usdcad_multivariate_service,
)

service = build_usdcad_multivariate_service()

print("USDCAD service loaded.")

Task was destroyed but it is pending!
task: <Task pending name='Task-51' coro=<LoggingWorker._worker_loop() running at /home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:122> wait_for=<Future pending cb=[Task.task_wakeup()]>>


USDCAD service loaded.


In [23]:
# Cell: Forecast task

from aieng.forecasting.evaluation.task import ForecastingTask

task = ForecastingTask(
    task_id=f"usdcad_logret_{HORIZON}b",
    target_series_id=target_series_id,
    horizons=[HORIZON],
    frequency="B",
    description=(
        f"USD/CAD cumulative log return, "
        f"{HORIZON} business days ahead."
    ),
)

print(task)

task_id='usdcad_logret_5b' target_series_id='usdcad_logret_5b' horizons=[5] frequency='B' description='USD/CAD cumulative log return, 5 business days ahead.' payload_type='continuous' categories=None resolution_fn='observed_value_at_resolution_timestamp'


In [14]:
# Cell: Build experiment contexts

def build_experiment_context(experiment_name):
    settings = EXPERIMENTS[experiment_name]

    context = {
        "target_series_id": f"usdcad_logret_{HORIZON}b",
        "horizon": HORIZON,
        "model": AGENT_MODEL,
    }

    if settings["use_covariates"]:
        context["covariate_series_ids"] = DEFAULT_COVARIATE_SERIES_IDS
    else:
        context["covariate_series_ids"] = []

    return context


experiment_contexts = {
    name: build_experiment_context(name)
    for name in EXPERIMENTS
}

experiment_contexts

{'baseline': {'target_series_id': 'usdcad_logret_5b',
  'horizon': 5,
  'model': 'gemini-3.1-flash-lite-preview',
  'covariate_series_ids': []},
 'fred_covariates': {'target_series_id': 'usdcad_logret_5b',
  'horizon': 5,
  'model': 'gemini-3.1-flash-lite-preview',
  'covariate_series_ids': ['usdcad_log_ret_1b_l1b',
   'fed_funds_level_l1b',
   'ust2y_level_l1b',
   'ust10y_level_l1b',
   'ust2y10y_spread_l1b',
   'cpi_mom_logdiff_l1b',
   'unemployment_rate_l1b',
   'oil_log_ret_1b_l1b']},
 'agentic_search': {'target_series_id': 'usdcad_logret_5b',
  'horizon': 5,
  'model': 'gemini-3.1-flash-lite-preview',
  'covariate_series_ids': ['usdcad_log_ret_1b_l1b',
   'fed_funds_level_l1b',
   'ust2y_level_l1b',
   'ust10y_level_l1b',
   'ust2y10y_spread_l1b',
   'cpi_mom_logdiff_l1b',
   'unemployment_rate_l1b',
   'oil_log_ret_1b_l1b']}}

In [15]:
# Cell: Run one experimental forecast

def run_experiment(experiment_name, context):
    """
    Run one USDCAD forecast under a specific experimental condition.
    """

    predictor = agents[experiment_name]

    settings = EXPERIMENTS[experiment_name]

    print("=" * 70)
    print(f"EXPERIMENT: {experiment_name}")
    print(settings["description"])
    print("=" * 70)

    # Use the same forecasting interface as the existing starter notebook.
    result = predictor.predict(
        task=task,
        context=context,
    )

    return result

In [25]:
RUN_AGENT = True

In [26]:
# Cell: Run experiments

if RUN_AGENT:

    results = {}

    for name, predictor in agents.items():

        print("\n" + "=" * 70)
        print(f"RUNNING: {name}")
        print(EXPERIMENTS[name]["description"])
        print("=" * 70)

        result = predictor.predict(
            task,
            ctx,
        )[0]

        results[name] = result

        fc = result.payload

        print(
            f"Point forecast: "
            f"{fc.point_forecast:+.6f} "
            f"({fc.point_forecast * 100:+.3f}%)"
        )

        print(
            f"80% interval: "
            f"[{fc.quantiles[0.10]:+.6f}, "
            f"{fc.quantiles[0.90]:+.6f}]"
        )

else:
    print("RUN_AGENT=False — no agent calls made.")


RUNNING: baseline
Target history only
Point forecast: -0.004200 (-0.420%)
80% interval: [-0.017500, +0.008500]

RUNNING: fred_covariates
Target history + FRED covariates
Point forecast: -0.005500 (-0.550%)
80% interval: [-0.017500, +0.006500]

RUNNING: agentic_search
Target history + FRED covariates + search
Point forecast: +0.001200 (+0.120%)
80% interval: [-0.004200, +0.008500]


In [27]:
# Cell: Compare forecasts with realized outcome

future = full[
    full["timestamp"] >= AS_OF + pd.offsets.BDay(HORIZON)
]

if future.empty:
    print("Actual outcome is not available yet.")

else:
    actual = float(future["value"].iloc[0])

    print("=" * 70)
    print(f"ACTUAL USDCAD {HORIZON}-BUSINESS-DAY RETURN")
    print("=" * 70)

    print(
        f"Actual: {actual:+.6f} "
        f"({actual * 100:+.3f}%)"
    )

    rows = []

    for name, pred in results.items():

        fc = pred.payload

        point = fc.point_forecast
        lo = fc.quantiles[0.10]
        hi = fc.quantiles[0.90]

        error = point - actual
        abs_error = abs(error)

        direction_correct = (
            (point >= 0 and actual >= 0)
            or
            (point < 0 and actual < 0)
        )

        inside_interval = lo <= actual <= hi

        rows.append({
            "agent": name,
            "forecast": point,
            "actual": actual,
            "error": error,
            "absolute_error": abs_error,
            "direction_correct": direction_correct,
            "80pct_coverage": inside_interval,
            "lower_80": lo,
            "upper_80": hi,
        })

    evaluation = pd.DataFrame(rows)

    display(
        evaluation[
            [
                "agent",
                "forecast",
                "actual",
                "error",
                "absolute_error",
                "direction_correct",
                "80pct_coverage",
            ]
        ]
    )

ACTUAL USDCAD 5-BUSINESS-DAY RETURN
Actual: -0.005211 (-0.521%)


,agent,forecast,actual,error,absolute_error,direction_correct,80pct_coverage
0,baseline,-0.0042,-0.005211,0.001011,0.001011,True,True
1,fred_covariates,-0.0055,-0.005211,-0.000289,0.000289,True,True
2,agentic_search,0.0012,-0.005211,0.006411,0.006411,False,False
